# Demo reset (lightweight re-baseline)

Returns the environment to the **post-setup baseline** so a fresh end-to-end run actually performs:

- **dev**: set default back to `V1`, delete `V2`, and remove the `ACCOUNT_RISK` feature view (added live in Act 4) — keeps the ABT, the `ACCOUNT_PROFILE` feature view, and V1.
- **prod**: drop the promoted model, the `PREDICTIONS` table, and the batch task.
- **preserved**: all data, dev V1, `ACCOUNT_PROFILE`, and the prod feature views (promotion re-registers those with `overwrite=True`).

After this: **Retrain** recreates dev `V2`, and **Promote** repopulates prod + predictions.

**Safety:** `DRY_RUN = True` by default — run top to bottom to preview, then set `DRY_RUN = False` and re-run to execute. Uses a session that can assume `ACCOUNTADMIN` and `ML_DEV_ROLE`.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Inlined names (mirror config.py) so the notebook is self-contained in a Workspace.
DEV_DATABASE     = "ML_FRAUD_DEV_SANDBOX"
PROD_DATABASE    = "ML_FRAUD_PRODUCTION"
REGISTRY_SCHEMA  = "ML"
ANALYTICS_SCHEMA = "ANALYTICS"
MODEL_NAME       = "AML_FRAUD_GBM"
KEEP_VERSION     = "V1"   # baseline model kept in dev
DROP_VERSION     = "V2"   # candidate the demo will recreate
PREDICTIONS      = f"{PROD_DATABASE}.{ANALYTICS_SCHEMA}.PREDICTIONS"
BATCH_TASK       = f"{PROD_DATABASE}.{ANALYTICS_SCHEMA}.SCORE_BATCH_TASK"
PROD_MODEL       = f"{PROD_DATABASE}.{REGISTRY_SCHEMA}.{MODEL_NAME}"

# SAFETY: preview only by default. Review the plan below, then set DRY_RUN = False and re-run to execute.
DRY_RUN = True

def run(sql):
    print(("[DRY-RUN] " if DRY_RUN else "[EXEC]    ") + sql)
    if not DRY_RUN:
        session.sql(sql).collect()

print("DRY_RUN =", DRY_RUN)

In [ ]:
# Current state (before)
def versions(db):
    try:
        return [r["name"] for r in session.sql(
            f"SHOW VERSIONS IN MODEL {db}.{REGISTRY_SCHEMA}.{MODEL_NAME}").collect()]
    except Exception as e:
        return f"(none: {type(e).__name__})"

def count(fqn):
    try:
        return session.sql(f"SELECT COUNT(*) AS C FROM {fqn}").collect()[0]["C"]
    except Exception:
        return None

print("dev model versions: ", versions(DEV_DATABASE))
print("prod model versions:", versions(PROD_DATABASE))
print("prod predictions:   ", count(PREDICTIONS))

In [ ]:
# [dev] set default back to V1, delete V2, and remove the ACCOUNT_RISK feature view
# so the feature store returns to its post-setup baseline (only ACCOUNT_PROFILE).
# Keeps the ABT and V1. Act 4 re-registers ACCOUNT_RISK (overwrite=True).
from snowflake.ml.registry import Registry
from snowflake.ml.feature_store import FeatureStore, CreationMode
session.sql("USE ROLE ML_DEV_ROLE").collect()   # owns the dev model + feature store

if DRY_RUN:
    print(f"[DRY-RUN] set dev {MODEL_NAME} default -> {KEEP_VERSION}; delete version {DROP_VERSION}")
    print("[DRY-RUN] delete dev feature view ACCOUNT_RISK / V1")
else:
    reg = Registry(session=session, database_name=DEV_DATABASE, schema_name=REGISTRY_SCHEMA)
    m = reg.get_model(MODEL_NAME)
    m.default = KEEP_VERSION            # must reset default before deleting V2
    try:
        m.delete_version(DROP_VERSION)
        print(f"[EXEC]    dev {DROP_VERSION} deleted; default now {KEEP_VERSION}")
    except Exception as e:
        print("delete version:", type(e).__name__, str(e)[:120])
    # remove ACCOUNT_RISK so it's absent before Act 4 (the feature-store schema must be current)
    session.sql(f"USE SCHEMA {DEV_DATABASE}.FEATURE_STORE").collect()
    try:
        fs = FeatureStore(session=session, database=DEV_DATABASE, name="FEATURE_STORE",
                          default_warehouse="CORTEX_CODE_WH", creation_mode=CreationMode.FAIL_IF_NOT_EXIST)
        fs.delete_feature_view(fs.get_feature_view("ACCOUNT_RISK", "V1"))
        print("[EXEC]    dev feature view ACCOUNT_RISK deleted (baseline: ACCOUNT_PROFILE only)")
    except Exception as e:
        print("delete ACCOUNT_RISK:", type(e).__name__, str(e).splitlines()[0][:120])

In [ ]:
# [prod] drop the promoted model, predictions table, and batch task.
# Feature views are left in place - promotion re-registers them with overwrite=True.
session.sql("USE ROLE ACCOUNTADMIN").collect()
try:
    session.sql("USE SECONDARY ROLES ALL").collect()   # activate owner role (ML_DEPLOY_SVC) for DROP
except Exception:
    pass

run(f"DROP TASK IF EXISTS {BATCH_TASK}")
run(f"DROP TABLE IF EXISTS {PREDICTIONS}")
run(f"DROP MODEL IF EXISTS {PROD_MODEL}")

In [ ]:
# Verify baseline (meaningful after DRY_RUN = False)
print("dev model versions: ", versions(DEV_DATABASE), "  (expect ['V1'])")
print("prod model versions:", versions(PROD_DATABASE), "  (expect none)")
print("prod predictions:   ", count(PREDICTIONS), "  (expect None)")
try:
    fvs = [r["name"] for r in session.sql(f"SHOW DYNAMIC TABLES IN SCHEMA {DEV_DATABASE}.FEATURE_STORE").collect()]
    print("dev feature views:  ", fvs, "  (expect only ACCOUNT_PROFILE$V1)")
except Exception:
    pass
print("\nBaseline ready: Retrain recreates dev V2 + ACCOUNT_RISK; Promote repopulates prod + predictions.")